# Educational attainment → ITL2 (APS counts, with Census robustness check)

Builds **qualification counts per ITL2 region per year (2004–2025)** and saves `clean/education_itl2.csv`; plus Census benchmarks in `clean/education_census_itl2.csv`.

**Source (main): Annual Population Survey, Nomis dataset NM_17_5** — the same dataset and pipeline as `labour_market.ipynb`. APS stores *% variables*; the **counts are their Numerator and Denominator measures**. For each region we keep, for people aged **16–64**:

| count | 2004–2021 (NVQ) | 2022–2025 (RQF) |
|---|---|---|
| level 4+ (degree level) | var 290 `% with NVQ4+` | var 1902 `% with RQF4+` |
| level 3+ | var 720 `% with NVQ3+` | var 2010 `% with RQF3+` |
| level 2+ | var 721 `% with NVQ2+` | var 2011 `% with RQF2+` |
| population 16–64 | denominator of the above | same |

`n_below_level2` = population − level 2+ (a standard low-skills measure). We don't use "no qualifications": it is suppressed for up to ~56% of LAs, so summing it would silently undercount.

**Method:** Jan–Dec periods only; LA → crosswalk → spine; **sum counts to ITL2**, derive shares at ITL2. An LA-year is used only if the population and all three counts are published (others listed in `aps_las_missing`). Northern Ireland: no LA-level APS on Nomis, so TLN0 uses the NI country figure.

**⚠ Classification break in 2022.** APS moved from NVQ to RQF levels, with no year published on both. Level 4+ is nearly continuous; **level 3+ jumps ~5 points nationally** (reclassification, not real change). We do **not** splice: the `qual_framework` column marks each year. With year fixed effects the national jump is absorbed; §3 measures how much the jump differs *between regions*, which is what could bias panel estimates.

**Robustness (Census):** level 4+ shares from the 2001, 2011 and 2021 Censuses — full counts, no sampling noise — summed to ITL2 the same way and compared with the APS (§5). Age bases differ (Census 16–74 / 16+ vs APS 16–64), so we compare **patterns across regions and changes**, not levels.

All source URLs are defined below and printed when run.

In [1]:
import io, time, urllib.request
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
SPINE     = BASE / "clean" / "geography_spine.csv"
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
RAW_DIR   = BASE / "raw" / "education"
OUT       = BASE / "clean" / "education_itl2.csv"
OUT_CENSUS = BASE / "clean" / "education_census_itl2.csv"

NOMIS = "https://www.nomisweb.co.uk/api/v01/dataset"
FRAMEWORKS = {"NVQ": {"level4plus": 290, "level3plus": 720, "level2plus": 721},     # APS 2004-2021
              "RQF": {"level4plus": 1902, "level3plus": 2010, "level2plus": 2011}}  # APS 2022+
VARS = ",".join(str(v) for f in FRAMEWORKS.values() for v in f.values())
APS_SELECT = "select=date,geography_code,variable,measures_name,obs_value"

URLS = {
    "APS, local authorities (April 2023)": f"{NOMIS}/NM_17_5.data.csv?geography=TYPE424&variable={VARS}&measures=21001,21002&date=first-latest&{APS_SELECT}",
    "APS, Northern Ireland (country)":     f"{NOMIS}/NM_17_5.data.csv?geography=N92000002&variable={VARS}&measures=21001,21002&date=first-latest&{APS_SELECT}",
    "APS, ITL2 direct (validation, 2025)":  f"{NOMIS}/NM_17_5.data.csv?geography=TYPE420&variable={VARS}&measures=21001,21002&date=first-latest&{APS_SELECT}",
    "Census 2001 KS013 (E&W, 16-74)":       f"{NOMIS}/NM_1619_1.data.csv?geography=TYPE464&cell=0,5&measures=20100&select=geography_code,cell,obs_value",
    "Census 2011 QS501UK (GB LAs, 16+)":    f"{NOMIS}/NM_1564_1.data.csv?geography=TYPE464&cell=0,6&measures=20100&select=geography_code,cell,obs_value",
    "Census 2011 QS501UK (NI districts)":   f"{NOMIS}/NM_1564_1.data.csv?geography=TYPE261&cell=0,6&measures=20100&select=geography_code,cell,obs_value",
    "Census 2021 TS067 (E&W, 16+)":         f"{NOMIS}/NM_2084_1.data.csv?geography=TYPE424&c2021_hiqual_8=0,6&measures=20100&select=geography_code,c2021_hiqual_8,obs_value",
}
for k, u in URLS.items():
    print(f"{k}:\n  {u}")

spine = pd.read_csv(SPINE)
assert spine.la_code.is_unique
lad2itl   = dict(zip(spine.la_code, spine.itl2_code))
itl2_name = dict(zip(spine.itl2_code, spine.itl2_name))
remap = dict(zip(*[pd.read_csv(CROSSWALK, dtype=str)[c] for c in ("old_code","new_code")]))
RAW_DIR.mkdir(parents=True, exist_ok=True)

def nomis(url, tag=None, limit=25000):
    """Fetch a Nomis CSV, paging past the 25,000-row cap; optionally save raw (dated)."""
    frames, offset = [], 0
    while True:
        for k in range(3):
            try:
                with urllib.request.urlopen(f"{url}&recordlimit={limit}&recordoffset={offset}", timeout=300) as r:
                    part = pd.read_csv(io.BytesIO(r.read()), encoding="utf-8-sig"); break
            except Exception as e:
                if k == 2: raise
                time.sleep(5)
        frames.append(part)
        if len(part) < limit: break
        offset += limit
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip().upper() for c in df.columns]
    if tag:
        df.to_csv(RAW_DIR / f"nomis_{tag}_{date.today().isoformat()}.csv", index=False)
    return df

APS, local authorities (April 2023):
  https://www.nomisweb.co.uk/api/v01/dataset/NM_17_5.data.csv?geography=TYPE424&variable=290,720,721,1902,2010,2011&measures=21001,21002&date=first-latest&select=date,geography_code,variable,measures_name,obs_value
APS, Northern Ireland (country):
  https://www.nomisweb.co.uk/api/v01/dataset/NM_17_5.data.csv?geography=N92000002&variable=290,720,721,1902,2010,2011&measures=21001,21002&date=first-latest&select=date,geography_code,variable,measures_name,obs_value
APS, ITL2 direct (validation, 2025):
  https://www.nomisweb.co.uk/api/v01/dataset/NM_17_5.data.csv?geography=TYPE420&variable=290,720,721,1902,2010,2011&measures=21001,21002&date=first-latest&select=date,geography_code,variable,measures_name,obs_value
Census 2001 KS013 (E&W, 16-74):
  https://www.nomisweb.co.uk/api/v01/dataset/NM_1619_1.data.csv?geography=TYPE464&cell=0,5&measures=20100&select=geography_code,cell,obs_value
Census 2011 QS501UK (GB LAs, 16+):
  https://www.nomisweb.co.uk/api/v01

## 1. APS counts by LA → ITL2

Per LA-year, pick the framework's variables (NVQ ≤ 2021, RQF ≥ 2022), keep numerators and the population denominator, and sum to ITL2.

In [2]:
def aps_wide(raw):
    """Long APS extract -> one row per geography x calendar year with counts for the right framework."""
    raw = raw[raw.DATE.astype(str).str.endswith("-12")].copy()                    # Jan-Dec periods only
    raw["year"] = raw.DATE.astype(str).str[:4].astype(int)
    raw["framework"] = np.where(raw.year <= 2021, "NVQ", "RQF")
    var2key = {(fw, v): k for fw, d in FRAMEWORKS.items() for k, v in d.items()}
    raw["key"] = [var2key.get((fw, v)) for fw, v in zip(raw.framework, raw.VARIABLE)]
    raw = raw.dropna(subset=["key"])
    w = raw.pivot_table(index=["GEOGRAPHY_CODE","year","framework"], columns=["key","MEASURES_NAME"],
                        values="OBS_VALUE", aggfunc="first")
    den = w.xs("Denominator", axis=1, level=1)
    spread = (den.max(axis=1) - den.min(axis=1)).max()
    assert spread <= 1000, f"denominators differ across variables by {spread}"
    out = pd.DataFrame({"pop_16_64": w[("level4plus","Denominator")]})
    for k in FRAMEWORKS["NVQ"]:
        out[f"n_{k}"] = w[(k, "Numerator")]
    return out.reset_index().rename(columns={"GEOGRAPHY_CODE": "geo"})

la = aps_wide(nomis(URLS["APS, local authorities (April 2023)"], tag="aps_quals_la"))
la["la_code"]   = la.geo.astype(str).str.strip().replace(remap)
la["itl2_code"] = la.la_code.map(lad2itl)
print("LA codes not in spine (dropped):", sorted(la.loc[la.itl2_code.isna(), "la_code"].unique()) or "none")
la = la.dropna(subset=["itl2_code"])
assert not la.dropna(subset=["n_level4plus"]).duplicated(["la_code","year"]).any(), "old + new LA codes both populated"

COUNTS = ["pop_16_64","n_level4plus","n_level3plus","n_level2plus"]
complete = la[COUNTS].notna().all(axis=1)
missing = la.loc[~complete].groupby(["itl2_code","year"]).la_code.apply(lambda s: ",".join(sorted(s)))
print(f"LA-years dropped (a count suppressed): {(~complete).sum()} of {len(la)}")

edu = la[complete].groupby(["itl2_code","year","framework"])[COUNTS].sum().reset_index()
edu["aps_las_missing"] = edu.set_index(["itl2_code","year"]).index.map(missing).fillna("")

ni = aps_wide(nomis(URLS["APS, Northern Ireland (country)"], tag="aps_quals_ni")).drop(columns="geo")
edu = pd.concat([edu, ni.assign(itl2_code="TLN0", aps_las_missing="")], ignore_index=True)
print(f"APS: {edu.itl2_code.nunique()} regions x {edu.year.nunique()} years ({edu.year.min()}-{edu.year.max()})")

LA codes not in spine (dropped): none
LA-years dropped (a count suppressed): 10 of 7660


APS: 46 regions x 22 years (2004-2025)


## 2. Validation against Nomis's own ITL2 figures

Nomis publishes APS at ITL2 (Jan 2025) only from 2025; our LA sums should reproduce it.

In [3]:
direct = aps_wide(nomis(URLS["APS, ITL2 direct (validation, 2025)"])).dropna(subset=["pop_16_64"])
cmp = edu.merge(direct, left_on=["itl2_code","year"], right_on=["geo","year"], suffixes=("","_direct"))
for c in COUNTS:
    gap = 100 * (cmp[c] / cmp[f"{c}_direct"] - 1)
    worst = cmp.loc[gap.abs().idxmax()]
    print(f"{c:14s} LA-sum vs direct ITL2 ({len(cmp)} region-years): median {gap.median():+.2f}%, "
          f"max |gap| {gap.abs().max():.1f}% ({worst.itl2_code}; LAs missing: {worst.aps_las_missing or 'none'})")
print("Largest gaps come from LAs dropped for suppression, flagged in aps_las_missing.")

pop_16_64      LA-sum vs direct ITL2 (45 region-years): median +0.00%, max |gap| 5.7% (TLM2; LAs missing: S12000023,S12000027)
n_level4plus   LA-sum vs direct ITL2 (45 region-years): median +0.00%, max |gap| 4.0% (TLM2; LAs missing: S12000023,S12000027)
n_level3plus   LA-sum vs direct ITL2 (45 region-years): median +0.00%, max |gap| 5.2% (TLM2; LAs missing: S12000023,S12000027)
n_level2plus   LA-sum vs direct ITL2 (45 region-years): median +0.00%, max |gap| 5.9% (TLM2; LAs missing: S12000023,S12000027)
Largest gaps come from LAs dropped for suppression, flagged in aps_las_missing.


## 3. Derive shares; size of the 2022 break by region

Shares are computed at ITL2 from the counts. For the break: each region's 2021→2022 change minus its own average annual change over 2016–2021 ("excess jump"). The national average excess is absorbed by year fixed effects; the **spread across regions** is what matters — compared with the same statistic in ordinary years (2017–2021), i.e. normal APS sampling noise.

In [4]:
edu["n_below_level2"] = edu.pop_16_64 - edu.n_level2plus
for k in ["level4plus","level3plus","level2plus","below_level2"]:
    edu[f"share_{k}"] = 100 * edu[f"n_{k}"] / edu.pop_16_64
edu = edu.rename(columns={"framework": "qual_framework"})

s = edu.pivot_table(index="itl2_code", columns="year", values=["share_level4plus","share_level3plus"])
def excess(k, y):
    """Region's change into year y minus its mean annual change over the 5 years before."""
    trend = (s[k][y - 1] - s[k][y - 6]) / 5
    return (s[k][y] - s[k][y - 1]) - trend

print("Excess jump by region, percentage points (change minus own 5-year trend):")
for k in ["share_level4plus","share_level3plus"]:
    br = excess(k, 2022)
    noise = np.mean([excess(k, y).std() for y in range(2016, 2022)])
    print(f"  {k:18s} 2022: national mean {br.mean():+.1f} pp, spread across regions sd {br.std():.1f} pp "
          f"(range {br.min():+.1f} to {br.max():+.1f}) | ordinary years 2016–2021: sd {noise:.1f} pp")

Excess jump by region, percentage points (change minus own 5-year trend):
  share_level4plus   2022: national mean +1.1 pp, spread across regions sd 2.6 pp (range -5.0 to +6.4) | ordinary years 2016–2021: sd 1.8 pp
  share_level3plus   2022: national mean +4.8 pp, spread across regions sd 2.7 pp (range -1.1 to +9.5) | ordinary years 2016–2021: sd 1.9 pp


## 4. Save

In [5]:
edu.insert(1, "itl2_name", edu.itl2_code.map(itl2_name))
cols = ["itl2_code","itl2_name","year","qual_framework","pop_16_64","n_level4plus","n_level3plus","n_level2plus",
        "n_below_level2","share_level4plus","share_level3plus","share_level2plus","share_below_level2","aps_las_missing"]
edu = edu[cols].sort_values(["year","itl2_code"]).reset_index(drop=True)
edu.to_csv(OUT, index=False, float_format="%.4f")
print(f"Saved -> {OUT}  ({len(edu)} rows)")
uk = edu.groupby("year")[["n_level4plus","n_level3plus","pop_16_64"]].sum()
print("\nUK share with level 4+ / level 3+ (%):")
print((100 * uk[["n_level4plus","n_level3plus"]].div(uk.pop_16_64, axis=0)).round(1).T.to_string())

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/education_itl2.csv  (1012 rows)

UK share with level 4+ / level 3+ (%):
year          2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
n_level4plus  26.0  26.5  27.4  28.5  28.5  29.7  31.1  32.6  34.0  34.9  35.8  36.9  38.0  38.4  39.1  40.2  43.0  43.5  45.5  47.1  47.1  48.4
n_level3plus  43.7  44.3  45.2  46.1  46.0  47.2  48.8  50.5  52.9  53.7  54.8  55.5  56.6  56.9  57.6  58.4  61.2  61.4  66.7  67.7  67.5  68.9


## 5. Census robustness check

Census level 4+ counts by LA, summed to ITL2 (2001 and 2021: England & Wales; 2011: UK — NI's pre-2015 districts all sit in TLN0). Compared with the APS share in the same year (2004 for Census 2001, the first APS year): correlation across regions, and for 2011→2021 the correlation of *changes*.

In [6]:
def census(url_keys, year, base, cell_col, tag):
    d = pd.concat([nomis(URLS[k], tag=f"census{year}_{i}") for i, k in enumerate(url_keys)], ignore_index=True)
    d["la_code"] = d.GEOGRAPHY_CODE.astype(str).str.strip().replace(remap)
    d["itl2_code"] = np.where(d.la_code.str.startswith(("95", "N")), "TLN0", d.la_code.map(lad2itl))
    bad = sorted(d.loc[d.itl2_code.isna(), "la_code"].unique())
    assert not bad, f"census {year}: codes not mapped to ITL2: {bad[:10]}"
    w = d.pivot_table(index="itl2_code", columns=cell_col, values="OBS_VALUE", aggfunc="sum")
    return pd.DataFrame({"census_year": year, "age_base": base, "persons": w[0], "n_level4plus": w[next(c for c in w.columns if c != 0)]}).reset_index()

cen = pd.concat([
    census(["Census 2001 KS013 (E&W, 16-74)"], 2001, "16-74", "CELL", "ks013"),
    census(["Census 2011 QS501UK (GB LAs, 16+)", "Census 2011 QS501UK (NI districts)"], 2011, "16+", "CELL", "qs501uk"),
    census(["Census 2021 TS067 (E&W, 16+)"], 2021, "16+", "C2021_HIQUAL_8", "ts067"),
], ignore_index=True)
cen["share_level4plus"] = 100 * cen.n_level4plus / cen.persons
cen.insert(1, "itl2_name", cen.itl2_code.map(itl2_name))
cen.to_csv(OUT_CENSUS, index=False, float_format="%.4f")
print(f"Saved -> {OUT_CENSUS}  ({len(cen)} rows)")
print(cen.groupby("census_year").agg(regions=("itl2_code","nunique"), base=("age_base","first")).to_string())

aps = edu.set_index(["itl2_code","year"]).share_level4plus
print("\nCensus vs APS level 4+ share across regions:")
for cy, ay in [(2001, 2004), (2011, 2011), (2021, 2021)]:
    c = cen[cen.census_year == cy].set_index("itl2_code").share_level4plus
    a = aps.xs(ay, level="year").reindex(c.index)
    print(f"  Census {cy} vs APS {ay}: {len(c)} regions | correlation {c.corr(a):.3f} | "
          f"Census mean {c.mean():.1f}% vs APS {a.mean():.1f}%")
c11, c21 = (cen[cen.census_year == y].set_index("itl2_code").share_level4plus for y in (2011, 2021))
d_cen = (c21 - c11).dropna()
d_aps = (aps.xs(2021, level="year") - aps.xs(2011, level="year")).reindex(d_cen.index)
print(f"  Change 2011→2021 (E&W, {len(d_cen)} regions): correlation of changes {d_cen.corr(d_aps):.3f} | "
      f"mean change Census {d_cen.mean():+.1f} pp vs APS {d_aps.mean():+.1f} pp")

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/education_census_itl2.csv  (124 rows)
             regions   base
census_year                
2001              39  16-74
2011              46    16+
2021              39    16+

Census vs APS level 4+ share across regions:
  Census 2001 vs APS 2004: 39 regions | correlation 0.924 | Census mean 19.5% vs APS 25.5%
  Census 2011 vs APS 2011: 46 regions | correlation 0.945 | Census mean 26.8% vs APS 32.2%
  Census 2021 vs APS 2021: 39 regions | correlation 0.976 | Census mean 33.3% vs APS 41.8%
  Change 2011→2021 (E&W, 39 regions): correlation of changes 0.619 | mean change Census +6.4 pp vs APS +10.2 pp
